# S-1 拓展课：排序

这一节用一个很小的问题认识两种经典排序方法：**冒泡排序**和**插入排序**。

我们不急着背代码。先把数放到数轴上观察：如果要从小到大排列，越小的数应该越靠左，越大的数应该越靠右。排序算法就是把一串还没有排好的数，一步一步变成这个样子。

## 学习目标

学完本节，你应该能做到：

- 在数轴上判断一组数的大小顺序。
- 说出冒泡排序每一步为什么只比较相邻两个数。
- 说出插入排序为什么像整理手里的扑克牌。
- 读懂简单的排序代码，并能修改要排序的数字。

**思维视角：数形结合 + 枚举验证。**

数形结合，就是把数字放到数轴上看位置。枚举验证，就是让 Python 把每一步都列出来，检查我们的想法是不是正确。

## 在数轴上的数进行排序

先看这 4 个数：

$$3,\quad 2,\quad 0,\quad 1$$

如果把它们放到数轴上，位置从左到右应该是：

$$0,\quad 1,\quad 2,\quad 3$$

所以，**从小到大排序**可以理解为：把这些数移动到数轴上从左到右的位置。

In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")

import matplotlib.pyplot as plt

plt.rcParams["axes.unicode_minus"] = False


def draw_number_line(values, xmin=None, xmax=None, title="number line"):
    """把一组数画在数轴上。"""
    xmin = min(values) - 1 if xmin is None else xmin
    xmax = max(values) + 1 if xmax is None else xmax

    fig, ax = plt.subplots(figsize=(8, 1.8))
    ax.annotate("", xy=(xmax + 0.3, 0), xytext=(xmin - 0.3, 0), arrowprops={"arrowstyle": "->"})
    for x in range(int(xmin), int(xmax) + 1):
        ax.plot([x, x], [-0.06, 0.06], color="black", linewidth=1)
        ax.text(x, -0.2, str(x), ha="center", va="top", fontsize=10)

    for index, value in enumerate(values):
        ax.scatter(value, 0.18, s=180, color="#4C78A8", zorder=3)
        ax.text(value, 0.18, str(value), ha="center", va="center", color="white", weight="bold")
        ax.text(value, 0.42, f"item {index + 1}", ha="center", va="bottom", fontsize=9)

    ax.set_xlim(xmin - 0.5, xmax + 0.6)
    ax.set_ylim(-0.45, 0.7)
    ax.set_title(title)
    ax.axis("off")
    plt.show()


def draw_row(numbers, title="list"):
    """把列表画成一排格子，方便观察排序过程。"""
    fig, ax = plt.subplots(figsize=(1.2 * len(numbers), 1.2))
    for i, number in enumerate(numbers):
        rect = plt.Rectangle((i, 0), 0.9, 0.8, facecolor="#F2F4F8", edgecolor="#333333")
        ax.add_patch(rect)
        ax.text(i + 0.45, 0.4, str(number), ha="center", va="center", fontsize=16)
        ax.text(i + 0.45, -0.18, f"pos {i}", ha="center", va="top", fontsize=9)
    ax.set_xlim(-0.1, len(numbers))
    ax.set_ylim(-0.35, 1.0)
    ax.set_title(title)
    ax.axis("off")
    plt.show()


def draw_trace(states, title):
    """把若干个列表状态画在一起。"""
    fig, axes = plt.subplots(len(states), 1, figsize=(7, 1.1 * len(states)))
    if len(states) == 1:
        axes = [axes]

    for ax, (label, numbers) in zip(axes, states):
        for i, number in enumerate(numbers):
            rect = plt.Rectangle((i, 0), 0.82, 0.72, facecolor="#EAF2F8", edgecolor="#333333")
            ax.add_patch(rect)
            ax.text(i + 0.41, 0.36, str(number), ha="center", va="center", fontsize=13)
        ax.text(-0.25, 0.36, label, ha="right", va="center", fontsize=10)
        ax.set_xlim(-1.8, len(numbers))
        ax.set_ylim(-0.1, 0.85)
        ax.axis("off")

    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
nums = [3, 2, 0, 1]

draw_number_line(nums, xmin=-1, xmax=4, title="Put the numbers on a number line")
draw_row(nums, title="Original order: the computer sees a list")

数轴帮助我们知道正确答案，但 Python 看到的是一个**列表**：

```python
[3, 2, 0, 1]
```

列表有位置编号：第 0 个、第 1 个、第 2 个、第 3 个。

排序时，程序只能做一些很小的动作，比如：

- 比较两个数谁大谁小。
- 如果顺序反了，就交换它们。
- 或者把一个数拿出来，插到前面合适的位置。

## 冒泡排序：大的数一步步往右走

冒泡排序（bubble sort）的想法很直观：

1. 只看相邻的两个数。
2. 如果左边比右边大，就交换。
3. 从左到右扫一遍后，最大的数会被换到最右边。
4. 再扫下一遍，继续把剩下的最大数换到右边。

它叫“冒泡”，是因为大的数会像气泡一样，一点一点“浮”到右边。

In [ ]:
def bubble_sort(numbers):
    """用冒泡排序把 numbers 从小到大排列。"""
    numbers = numbers.copy()
    n = len(numbers)

    for pass_index in range(n - 1):
        for j in range(0, n - 1 - pass_index):
            if numbers[j] > numbers[j + 1]:
                numbers[j], numbers[j + 1] = numbers[j + 1], numbers[j]

    return numbers


print("sorted result:", bubble_sort(nums))

### 练习：自己完整实现冒泡排序

请补全下面的 `bubble_sort_practice`。要求：

- 不使用 Python 自带的 `sorted()` 或 `.sort()`。
- 每次只比较相邻两个数。
- 如果左边大于右边，就交换。
- 最后返回从小到大排好的新列表，不要改动原来的列表。

In [ ]:
bubble_test_data = [5, 1, 4, 2, 0]
bubble_expected = [0, 1, 2, 4, 5]


def bubble_sort_practice(numbers):
    result = numbers.copy()

    n = len(result)
    for pass_index in range(n - 1):
        for j in range(0, n - 1 - pass_index):
            if result[j] > result[j + 1]:
                result[j], result[j + 1] = result[j + 1], result[j]

    return result


print("test data:", bubble_test_data)
print("expected:", bubble_expected)
print("your result:", bubble_sort_practice(bubble_test_data))

assert bubble_sort_practice(bubble_test_data) == bubble_expected

为了看清楚冒泡排序每一轮发生了什么，我们再写一个带记录功能的版本。它和上面的简洁版本思路一样，只是多保存了每一轮结束后的列表状态。

In [ ]:
def bubble_sort_trace(numbers):
    """用冒泡排序排列 numbers，并记录每一轮结束后的状态。"""
    numbers = numbers.copy()
    n = len(numbers)
    states = [("start", numbers.copy())]

    for pass_index in range(n - 1):
        for j in range(0, n - 1 - pass_index):
            if numbers[j] > numbers[j + 1]:
                numbers[j], numbers[j + 1] = numbers[j + 1], numbers[j]
        states.append((f"pass {pass_index + 1}", numbers.copy()))

    return numbers, states


sorted_nums, bubble_states = bubble_sort_trace(nums)
print("sorted result:", sorted_nums)
draw_trace(bubble_states, "Bubble sort: after each pass")

只看每一轮结束还不够。下面把第一轮的每一次相邻比较列出来。

第一轮从 `[3, 2, 0, 1]` 开始：

- 比较 3 和 2，顺序反了，交换，变成 `[2, 3, 0, 1]`。
- 比较 3 和 0，顺序反了，交换，变成 `[2, 0, 3, 1]`。
- 比较 3 和 1，顺序反了，交换，变成 `[2, 0, 1, 3]`。

你会发现，最大的 3 已经到了最右边。下一轮就不用再动它了。

In [ ]:
first_pass = [3, 2, 0, 1]
first_pass_states = [("start", first_pass.copy())]

for j in range(len(first_pass) - 1):
    if first_pass[j] > first_pass[j + 1]:
        first_pass[j], first_pass[j + 1] = first_pass[j + 1], first_pass[j]
    first_pass_states.append((f"compare {j} and {j + 1}", first_pass.copy()))

draw_trace(first_pass_states, "Bubble sort: first pass")

## 插入排序：把新来的数插到合适位置

插入排序（insertion sort）像整理手里的扑克牌。

假设左边已经排好了。每次从右边拿一个新数，和左边已经排好的数比较，然后把它插到合适的位置。

例如 `[3, 2, 0, 1]`：

- 先认为 `[3]` 已经排好。
- 拿到 2，把它插到 3 前面，得到 `[2, 3]`。
- 拿到 0，把它插到最前面，得到 `[0, 2, 3]`。
- 拿到 1，把它插到 0 和 2 中间，得到 `[0, 1, 2, 3]`。

In [ ]:
def insertion_sort(numbers):
    """用插入排序把 numbers 从小到大排列。"""
    numbers = numbers.copy()

    for i in range(1, len(numbers)):
        key = numbers[i]
        j = i - 1

        while j >= 0 and numbers[j] > key:
            numbers[j + 1] = numbers[j]
            j -= 1

        numbers[j + 1] = key

    return numbers


print("sorted result:", insertion_sort(nums))

### 练习：自己完整实现插入排序

请补全下面的 `insertion_sort_practice`。要求：

- 不使用 Python 自带的 `sorted()` 或 `.sort()`。
- 从第 2 个数开始，依次把新数插入左边已经排好的部分。
- 移动比新数大的旧数，给新数让出位置。
- 最后返回从小到大排好的新列表，不要改动原来的列表。

In [ ]:
insertion_test_data = [3, -1, 2, -5, 0]
insertion_expected = [-5, -1, 0, 2, 3]


def insertion_sort_practice(numbers):
    result = numbers.copy()

    for i in range(1, len(result)):
        key = result[i]
        j = i - 1

        while j >= 0 and result[j] > key:
            result[j + 1] = result[j]
            j -= 1

        result[j + 1] = key

    return result


print("test data:", insertion_test_data)
print("expected:", insertion_expected)
print("your result:", insertion_sort_practice(insertion_test_data))

assert insertion_sort_practice(insertion_test_data) == insertion_expected

为了观察每次“插入”之后列表变成什么样，我们再写一个带记录功能的版本。这个版本会保存每次插入后的状态。

In [ ]:
insertion_test_data = [3, -1, 2, -5, 0]
insertion_expected = [-5, -1, 0, 2, 3]

def insertion_sort_trace(numbers):
    """用插入排序排列 numbers，并记录每次插入后的状态。"""
    numbers = numbers.copy()
    states = [("start", numbers.copy())]

    for i in range(1, len(numbers)):
        key = numbers[i]
        j = i - 1

        while j >= 0 and numbers[j] > key:
            numbers[j + 1] = numbers[j]
            j -= 1

        numbers[j + 1] = key
        states.append((f"insert {key}", numbers.copy()))

    return numbers, states


sorted_nums, insertion_states = insertion_sort_trace(nums)
print("sorted result:", sorted_nums)
draw_trace(insertion_states, "Insertion sort: after each insertion")

## 两种方法有什么不同

| 方法 | 每一步在做什么 | 像什么 | 适合怎样理解 |
|---|---|---|---|
| 冒泡排序 | 比较相邻两个数，顺序反了就交换 | 大的气泡往右冒 | 观察“最大值逐轮归位” |
| 插入排序 | 拿出一个新数，插到前面已经排好的部分 | 整理手里的牌 | 观察“左边始终有序” |

这两种方法最后都能得到 `[0, 1, 2, 3]`。不同的是，它们走向答案的路线不一样。

In [ ]:
new_nums = [4, -1, 3, 0, -2]

print("original:", new_nums)
print("bubble sort:", bubble_sort(new_nums))
print("insertion sort:", insertion_sort(new_nums))

draw_number_line(new_nums, xmin=-3, xmax=5, title="A new group of numbers")

## 变式练习

请把上面代码中的 `new_nums` 改成下面几组数，先自己在纸上排，再运行 Python 检查：

1. `[5, 1, 4, 2]`
2. `[0, -3, 2, -1]`
3. `[2, 2, 1, 3]`

思考两个问题：

- 如果两个数一样大，排序后它们的大小关系有没有变？
- 如果一开始已经排好了，冒泡排序和插入排序还会做哪些检查？

<details>
<summary>参考思路</summary>

两个数一样大时，不需要交换。比如 `[2, 2, 1, 3]` 中的两个 2 大小相同，谁在前面都不影响从小到大的结果。

如果一开始已经排好了，程序仍然会比较一些数。因为程序不像人一样一眼看完整体，它要按规则一步一步确认。

</details>

 ## 小结反思

这一节把排序和数轴联系起来：从小到大，就是在数轴上从左到右。

冒泡排序的关键是：**相邻比较，必要时交换，最大值逐轮到右边。**

插入排序的关键是：**左边保持有序，新数插入到合适位置。**

学习算法时，不要只盯着最后答案。更重要的是观察每一步：程序为什么这样比较？为什么这样移动？这正是数学建模和计算思维的共同点。